# Visit overview: demo

Concept: `overview/concepts/Visit Mappings.md` (outside this repo), sections "identifying shifts" (dwell time, time spent at, inside and outside, transfers going to a location) and "Visit Overview".
Code: `ocelescope_module_location_based_ocpm.visits` (`dwell.py`, `overview.py`, `pipeline.py`) and `visit_mapping.pipeline.analyse_mapping`; this notebook only imports and calls functions.
Further detail: the docstrings and `docs/resources/visit-overview.md`.

What the notebook does:
1. loads the running example of the concept note,
2. computes dwell times and the time spent at locations, inside and outside $hl$,
3. aggregates transfers per category and per target (vehicle or external high-level location),
4. aggregates waiting times per processing activity,
5. builds one overview row per visit,
6. runs everything in one call, for the maximal visits (`analyse_visits`), for a mapping set (`analyse_mapping`) and for any visit set (`analyse_visit_set`).

Start Jupyter from the repository root with

```
uv run --with jupyter jupyter lab notebooks/visit_overview.ipynb
```

Conventions in all frames: one row per hu-event relation $(e, hu)$ unless stated otherwise; `null` is $\bot$; durations are polars durations (e.g. `10h`).
All functions take and return polars frames; they work on eager (`DataFrame`) and lazy (`LazyFrame`) frames alike.

In [ ]:
from pathlib import Path

import polars as pl

from ocelescope_module_location_based_ocpm.derivation.level_files import MAPPING_SCHEMA
from ocelescope_module_location_based_ocpm.visit_mapping.pipeline import analyse_mapping, derive_mapping_set
from ocelescope_module_location_based_ocpm.visits.context import location_context
from ocelescope_module_location_based_ocpm.visits.dwell import add_dwell_times, time_inside_outside, time_spent
from ocelescope_module_location_based_ocpm.visits.maximal import derive_maximal_visits, visit_relations
from ocelescope_module_location_based_ocpm.visits.overview import (
    processing_statistics,
    transfer_relations,
    transfer_statistics,
    transfer_targets,
    visit_overview,
)
from ocelescope_module_location_based_ocpm.visits.pipeline import analyse_visit_set, analyse_visits
from ocelescope_module_location_based_ocpm.visits.shifts import add_stops, transfers
from ocelescope_module_location_based_ocpm.waiting_times.pipeline import analyse_waiting_times

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "mapping_example"

pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(250)
pl.Config.set_fmt_table_cell_list_len(8)
pl.Config.set_fmt_str_lengths(40);

## 1. Data

The running example of the concept note (`data/mapping_example/`), $\kappa_{max} = 3$, $V = \{T\}$, $W = \{B1\}$. Event eK happens at hour K.

In [ ]:
KAPPA = 3
relations = pl.read_csv(DATA / "relations.csv", schema=MAPPING_SCHEMA)
context = location_context(pl.read_csv(DATA / "locations.csv"))
P = pl.read_csv(DATA / "processing_events.csv")["event"].to_list()
rel, maximal = derive_maximal_visits(relations, "loc", context.agg, KAPPA, processing=P)
vrel = add_stops(visit_relations(rel, maximal), context.vehicles)

## 2. Dwell times: `add_dwell_times(visit_relations, visits)`, `time_spent(dwell_relations, by)`, `time_inside_outside(dwell_relations, visits)`

$\delta_\nu(i) = time(\sigma[i+1]) - time(\sigma[i])$, and for the last relation $time(e_{out}) - time(\sigma[n])$; the handling unit is at $loc(\sigma[i])$ for $\delta_\nu(i)$, and the dwell times add up to $d(\nu)$.
`time_spent` sums them per visit and location (`by="loc"`: $d_x(\nu)$; `by="high_level"`: $d_{hl'}(\nu)$); `time_inside_outside` gives $d_{hl}(\nu)$ and $d(\nu) - d_{hl}(\nu)$.

Example b2's visit to A (e3–e11): 8 h inside, 1 h outside (on T at e8).

In [ ]:
dwell = add_dwell_times(vrel, maximal)
b2_a = maximal.filter((pl.col("hu") == "b2") & (pl.col("high_level") == "A"))["visit_id"].item()
display(dwell.filter(pl.col("visit_id") == b2_a).select("visit_id", "position", "event", "time", "loc", "high_level", "at_location", "dwell_time"))
display(time_spent(dwell, "loc").filter(pl.col("visit_id") == b2_a))
display(time_spent(dwell, "high_level").filter(pl.col("visit_id") == b2_a))
time_inside_outside(dwell, maximal).filter(pl.col("visit_id") == b2_a)

## 3. Transfers: `transfer_statistics(transfers)`, `transfer_relations(dwell, transfers)`, `transfer_targets(transfer_relations, location)`

Per visit and category: number of transfers, duration total / mean / min / max / median, length mean / min / max / median.
Per visit, category and target (the vehicle of a vehicle shift, the external high-level location of an external shift): transfers going to it, relations at it and time spent at it, total and min / max / median / mean per transfer.

In [ ]:
transfer_frame = transfers(dwell)
display(transfer_statistics(transfer_frame))
transfer_targets(transfer_relations(dwell, transfer_frame))

## 4. Processing activities: `processing_statistics(waiting_times)`

Per visit and processing activity: number of operations and waiting time total / min / max / median / mean (waiting times from `waiting_times.ipynb`).

In [ ]:
waiting = analyse_waiting_times(rel, "loc", maximal, processing=P, storage_units=context.storage_units)
processing_statistics(waiting.waiting_times)

## 5. One row per visit: `visit_overview(visits, inside_outside, statistics, waiting_summary, processing)`

Joins the visits with the inside/outside times, the statistics of vehicle and external shifts (as columns `vehicle_shift_...`, `external_shift_...`), the waiting summary and the number of processing activities.
`analyse_visit_set` (next section) builds it with the visit properties and the shift summary as well.

In [ ]:
visit_overview(
    maximal,
    time_inside_outside(dwell, maximal),
    transfer_statistics(transfer_frame),
    waiting.summary,
    processing_statistics(waiting.waiting_times),
).select("visit_id", "hu", "high_level", "first_event", "last_event", "duration", "time_inside", "time_outside", "vehicle_shift_n_transfers", "external_shift_n_transfers", "external_shift_duration_total", "total_waiting_time", "n_processing_activities")

## 6. Everything in one call

- `analyse_visits(relations, location, agg, vehicles, kappa, processing, ..., storage_units=(), with_waiting_times=True, **waiting_options)`: $\mathcal{V}_{max}$ and everything below;
- `analyse_visit_set(relations, visits, location, vehicles, processing, storage_units, vehicles_outside_external, with_waiting_times, agg, **waiting_options)`: the same for any visit set (relations with position and high_level);
- `analyse_mapping(mapping_analysis, location, vehicles, processing, storage_units, ...)`: the same for a mapping set from `derive_mapping_set`.

All return a `VisitAnalysis` with: relations, visits, visit_relations, transfers, naming_clashes, excursions, pairs, set_properties, set_metrics, set_metrics_per_hu, time_at_locations, time_at_high_levels, transfer_statistics, transfer_targets, waiting, processing_statistics, overview, rq4_violations.

In [ ]:
vmax = analyse_visits(relations, "loc", context.agg, context.vehicles, KAPPA, P, storage_units=context.storage_units)
display(vmax.set_metrics)
vmax.overview.select("visit_id", "hu", "high_level", "first_event", "last_event", "kind", "n_excursions", "n_stops", "n_external_shifts", "time_inside", "time_outside", "total_waiting_time")

The S6 mapping set with its shifts, waiting times and overview:

In [ ]:
s6 = derive_mapping_set(relations, "loc", context.agg, context.vehicles, KAPPA, P, "S6", context.storage_units)
vmap = analyse_mapping(s6, "loc", context.vehicles, P, context.storage_units)
display(vmap.set_metrics)
display(vmap.transfers.select("visit_id", "hu", "visit_high_level", "j", "from_event", "to_event", "category", "transfer_duration"))
vmap.overview.select("visit_id", "hu", "high_level", "first_event", "last_event", "kind", "time_inside", "time_outside", "n_stays", "n_vehicle_shifts", "n_external_shifts", "total_waiting_time", "n_processing_activities")

Any hand-written visit set (given by first and last event), here without waiting times:

In [ ]:
hand = pl.DataFrame({"hu": ["b1", "b1"], "high_level": ["A", "T"], "first_event": ["e3", "e13"], "last_event": ["e5", "e14"]})
own = analyse_visit_set(vmax.relations, hand, "loc", context.vehicles, P, with_waiting_times=False)
display(own.set_properties.filter(pl.col("hu") == "b1"))
own.overview.select("visit_id", "hu", "high_level", "first_event", "last_event", "kind", "duration", "time_inside", "time_outside")